In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
from urllib.parse import quote
from datetime import datetime, timedelta
import ast

# Constants and configurations
EA_URL = 'https://store.steampowered.com/search/?publisher=Electronic%20Arts&page='
LIST_TO_DROP = [977660, 988670, 1379260, 1262591, 1315480, 1315442, 988662, 1315435, 
                1315471, 1315434, 1315470, 1315430, 1315431, 1315443, 1315472, 
                1315441, 1315440, 1315432, 1315433, 1363591, 1262590]

# Scrape EA games from Steam search
def scrape_ea_games(page):
    url = EA_URL + str(page)
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    game_list = []
    games = soup.find_all('a', {'class': 'search_result_row'})
    for game in games:
        game_url = game['href']
        try:
            appid = game_url.split('/app/')[1].split('/')[0]
            if appid.isdigit():
                game_name = game.find('span', {'class': 'title'}).text.strip()
                game_list.append({'appid': appid, 'name': game_name})
        except IndexError:
            continue
    return game_list

# Fetch app details to filter games and DLCs
def get_steam_app_details(app_id):
    url = f"https://store.steampowered.com/api/appdetails?appids={app_id}"
    try:
        response = requests.get(url)
        data = response.json()
        if data[str(app_id)]['success']:
            return data[str(app_id)]['data'].get('type', 'unknown')
        return None
    except Exception as e:
        print(f"Error fetching data for app id {app_id}: {e}")
        return None

# Collect all EA games
def collect_ea_games():
    all_games = []
    page = 1
    while True:
        print(f"Scraping page {page}...")
        games = scrape_ea_games(page)
        if not games:
            break
        all_games.extend(games)
        time.sleep(1)
        page += 1
    return all_games

# Fetch reviews from Steam API
def get_steam_reviews(app_id, pages=100):
    base_reviews_url = f"https://store.steampowered.com/appreviews/{app_id}?json=1&num_per_page=20"
    reviews = []
    cursor = '*'  
    seen_cursors = set()

    for page in range(1, pages + 1):
        print(f"Fetching page {page} for app ID {app_id}...")
        encoded_cursor = quote(cursor).replace('+', '%2B')
        response = requests.get(f"{base_reviews_url}&cursor={encoded_cursor}")
        data = response.json()
        if not data.get('reviews'):
            break

        for review in data['reviews']:
            reviews.append({
                'recommended': 'Recommended' if review.get('voted_up', False) else 'Not Recommended',
                'date': time.strftime('%Y-%m-%d', time.localtime(review.get('timestamp_created'))),
                'comment': review.get('review', 'N/A'),
                'playtime_hours': review.get('author', {}).get('playtime_forever', 0) / 60,
                'votes_up': review.get('votes_up', 0)
            })

        cursor = data.get('cursor', '')
        if cursor in seen_cursors:
            break
        seen_cursors.add(cursor)
        if not cursor:
            break
        time.sleep(1)
    return reviews

# Get detailed game information
def get_steam_game_info(app_id):
    store_url = f"https://store.steampowered.com/api/appdetails?appids={app_id}&cc=us"
    store_response = requests.get(store_url)
    try:
        response_json = store_response.json()
        store_data = response_json.get(str(app_id), {}).get('data', None)
    except (ValueError, AttributeError) as e:
        print(f"Error parsing JSON for app ID {app_id}: {e}")
        return None

    if not store_data:
        return None

    game_name = store_data.get('name', 'N/A')
    price = store_data.get('price_overview', {}).get('final_formatted', 'Free')
    reviews = get_steam_reviews(app_id)
    return {
        'appid': app_id,
        'name': game_name,
        'price': price,
        'reviews': reviews
    }

# Collect game data
def collect_game_data(appid_list):
    all_games_info = []
    for app_id in appid_list:
        print(f"Fetching data for appid: {app_id}")
        game_info = get_steam_game_info(app_id)
        if game_info:
            all_games_info.append(game_info)
        time.sleep(1)
    return all_games_info

# Save data to file
def save_game_info_to_file(all_games_info, output_file='data/steam_reviews_data.csv'):
    flattened_data = []
    for game in all_games_info:
        flattened_data.append({
            'appid': game['appid'],
            'name': game['name'],
            'price': game['price'],
            #'reviews': [review['comment'] for review in game['reviews']],
            'review_dates': [review['date'] for review in game['reviews']],
            'recommended_status': [review['recommended'] for review in game['reviews']],
            'playtime_hours': [review['playtime_hours'] for review in game['reviews']]
        })

    df = pd.DataFrame(flattened_data)
    df.to_csv(output_file, index=False)
    print(f"File saved as {output_file}.")


def analyze_reviews(file_path):
    """
    Analyze reviews across all rows, combining and counting recommendations and not recommendations by date.
    """
    df = pd.read_csv(file_path, encoding='utf-8')

    # Initialize variables to hold all dates and recommendation statuses
    all_dates = []
    all_recommendations = []

    # Process each row to extract review dates and recommendations
    for _, row in df.iterrows():
        # Convert string representations of lists to actual lists
        row_dates = ast.literal_eval(row['review_dates'])
        row_recommendations = ast.literal_eval(row['recommended_status'])

        # Convert string dates to datetime.date objects
        row_dates_converted = [pd.to_datetime(date, format='%Y-%m-%d').date() for date in row_dates]

        # Append the data from this row to the combined lists
        all_dates.extend(row_dates_converted)
        all_recommendations.extend(row_recommendations)

    # Determine the range of dates
    oldest_date = min(all_dates)
    newest_date = max(all_dates)
    dates_list = [oldest_date + timedelta(days=i) for i in range((newest_date - oldest_date).days + 1)]

    # Map dates to indices for counting
    date_to_index = {date: idx for idx, date in enumerate(dates_list)}

    # Initialize counters for recommendations and not recommendations
    recommend = [0] * len(dates_list)
    not_recommend = [0] * len(dates_list)

    # Count recommendations and not recommendations by date
    for i in range(len(all_dates)):
        index = date_to_index.get(all_dates[i])
        if index is not None:
            if all_recommendations[i] == "Recommended":
                recommend[index] += 1
            elif all_recommendations[i] == "Not Recommended":
                not_recommend[index] += 1

    # Print the results
    print("Recommendation counts:", recommend)
    print("Not Recommendation counts:", not_recommend)

    # Return the lists
    return recommend, not_recommend, date_to_index

In [ ]:
# Execute the complete process
ea_games = collect_ea_games()
filtered_appids = [game['appid'] for game in ea_games if game['appid'] not in str(LIST_TO_DROP)]

In [ ]:
game_data = collect_game_data(filtered_appids[3:6])  # Limit to first 5 games for testing"

In [ ]:
save_game_info_to_file(game_data)

In [ ]:
recommend, not_recommend, date = analyze_reviews('data/steam_reviews_data.csv')

In [ ]:
date_list = list(date.keys())
date_list = [str(date) for date in date.keys()]

print(date_list)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(date_list, recommend, label='Recommended', color='green', align='center')
ax.bar(date_list, [-count for count in not_recommend], label='Not Recommended', color='red', align='center')
ax.set_xlabel('Date')
ax.set_ylabel('Count')
ax.set_title('Daily Recommendations vs. Not Recommendations')
plt.xticks(rotation=45)
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
print(len(date))

In [ ]:
import yfinance as yf

ticker_symbol = 'EA'
stock_data = yf.download(ticker_symbol, start=date_list[0], end=date_list.index[-1])

#show sm data
print(stock_data.head())
stock_data.to_csv('data/EA_stock_data.csv')

#show most recent data 
latest_date = stock_data.index[-1]
print("Most recent date for EA stock data:", latest_date)

In [ ]:
import pandas as pd

ea_stock_data = pd.read_csv('data/EA_stock_data.csv')
close_values = ea_stock_data['Close'].tolist()
date_stocks = ea_stock_data['Date'].tolist()
print(close_values)
print(date_stocks)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

ea_stock_data = pd.read_csv('data/EA_stock_data.csv')
ea_stock_data['Date'] = pd.to_datetime(ea_stock_data['Date'])
dates = ea_stock_data['Date']
close_values = ea_stock_data['Close']

plt.figure(figsize=(10, 6))
plt.plot(dates, close_values, marker='o', color='b', label='Close Price')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.title('Stock Close Prices Over Time')
plt.grid(True)
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()
